# 07 — Training Neural Networks I: Activations, Initialization, BatchNorm

**CS231n Lecture 6.** You can now build and differentiate any architecture. This
notebook is about the things that decide whether it *trains* — the setup
choices that turn a correct network into a working one.

The through-line: **keep activations at a healthy scale at every depth.** Too
small and they vanish; too large and they saturate. Every technique here
attacks that one problem from a different angle.

| You write | |
|---|---|
| `tanh_forward/backward`, `leaky_relu_forward/backward` | the alternatives to ReLU |
| `init_weights` | five schemes, including two that fail on purpose |
| `forward_stats` | the experiment that makes initialisation visible |
| `batchnorm_forward` | train and test mode — they differ, and it matters |
| `batchnorm_backward` | the simplified derivation |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cs231n_utils import (get_cifar10, rel_error, grade, show_solution, use,
                          eval_numerical_gradient_array, plot_series,
                          setup_plots, SERIES)

setup_plots()
np.random.seed(0)

## 1. Activation functions

| | Range | Zero-centred? | Saturates? | Verdict |
|---|---|---|---|---|
| **sigmoid** | (0, 1) | no | both ends | do not use in hidden layers |
| **tanh** | (-1, 1) | yes | both ends | better than sigmoid, still saturates |
| **ReLU** | [0, ∞) | no | negative side only | **the default** |
| **Leaky ReLU** | (-∞, ∞) | roughly | never | use if you suspect dying ReLU |

Sigmoid has three separate problems and it is worth being able to name all
three:

1. **Saturation kills gradients.** Local gradient peaks at 0.25 and is
   effectively 0 in the tails (notebook 04).
2. **Output is not zero-centred.** All outputs are positive, so for the next
   layer every input is positive, so every weight in a row gets a gradient of
   the same sign. The weights can only all-increase or all-decrease together,
   which forces a **zig-zag** path to the optimum.
3. **`exp` is expensive.** Minor next to the other two, but real.

ReLU fixes 1 and 3 outright. It does not fix 2 — and that is precisely the hole
batch normalisation fills.

In [ ]:
xr = np.linspace(-5, 5, 400)
acts = {
    'sigmoid':    (1 / (1 + np.exp(-xr)),              lambda s: s * (1 - s)),
    'tanh':       (np.tanh(xr),                        lambda s: 1 - s ** 2),
    'ReLU':       (np.maximum(0, xr),                  None),
    'Leaky ReLU': (np.where(xr > 0, xr, 0.1 * xr),     None),
}
grads = {
    'sigmoid': (1 / (1 + np.exp(-xr))) * (1 - 1 / (1 + np.exp(-xr))),
    'tanh': 1 - np.tanh(xr) ** 2,
    'ReLU': (xr > 0).astype(float),
    'Leaky ReLU': np.where(xr > 0, 1.0, 0.1),
}

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for i, (nm, (vals, _)) in enumerate(acts.items()):
    axes[0].plot(xr, vals, color=SERIES[i], label=nm)
    axes[1].plot(xr, grads[nm], color=SERIES[i], label=nm)
axes[0].set_title("Activation"); axes[1].set_title("Local gradient")
for ax in axes:
    ax.set_xlabel("x"); ax.legend(); ax.set_axisbelow(True)
axes[1].set_ylim(-0.1, 1.2)
plt.show()

### The dying ReLU problem

A ReLU unit whose input is negative for *every* example in the dataset outputs
zero always and receives zero gradient always. It is permanently dead — no
gradient can ever revive it. This happens when a large gradient step knocks the
bias very negative, which is why "10–20% of your ReLUs are dead" is a common
(and usually tolerable) diagnostic, and why a huge learning rate can silently
kill half a layer.

Leaky ReLU (`0.01x` for negative input) guarantees a nonzero gradient
everywhere, so units cannot die.

### TODO 1 — tanh and Leaky ReLU

For tanh, note $\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)$ — expressed via the
**output**, so cache the output, exactly like sigmoid.

In [ ]:
def tanh_forward(x):
    """-> (out, cache).  Cache the OUTPUT, not the input."""
    pass  # TODO

def tanh_backward(dout, cache):
    """-> dx"""
    pass  # TODO

def leaky_relu_forward(x, alpha=0.01):
    """-> (out, cache).  Cache what backward needs."""
    pass  # TODO

def leaky_relu_backward(dout, cache):
    """-> dx"""
    pass  # TODO

In [ ]:
for nm, fn in [('tanh_forward', tanh_forward), ('tanh_backward', tanh_backward),
               ('leaky_relu_forward', leaky_relu_forward),
               ('leaky_relu_backward', leaky_relu_backward)]:
    grade('nb07', nm, fn, verbose=False)

In [ ]:
tf = use('nb07', 'tanh_forward', tanh_forward)
tb = use('nb07', 'tanh_backward', tanh_backward)
lf = use('nb07', 'leaky_relu_forward', leaky_relu_forward)
lb = use('nb07', 'leaky_relu_backward', leaky_relu_backward)

xa = np.random.RandomState(2).randn(5, 4)
da = np.random.RandomState(3).randn(5, 4)
for nm, fw, bw in [('tanh', tf, tb), ('leaky relu', lf, lb)]:
    _, ca = fw(xa)
    print("%-11s dx rel error %.2e" % (nm, rel_error(
        bw(da, ca), eval_numerical_gradient_array(lambda t: fw(t)[0], xa.copy(), da))))

## 2. Weight initialization — the experiment that makes it obvious

### Why not zeros?

If every weight is zero, every hidden unit in a layer computes the identical
output, receives the identical gradient, and updates identically. They stay
identical forever. A 1000-unit layer behaves as a single unit. **This is the
symmetry-breaking problem, and it is why initialisation must be random.**

### Then how random?

Consider a layer output $y = \sum_{i=1}^{n} w_i x_i$ with independent, zero-mean
$w$ and $x$:

$$ \text{Var}(y) = n \,\text{Var}(w)\,\text{Var}(x) $$

For the activation scale to be *preserved* across a layer you need
$n\,\text{Var}(w) = 1$, i.e.

$$ \text{Var}(w) = \frac{1}{n_{in}} \qquad\Longrightarrow\qquad w \sim \frac{\mathcal{N}(0,1)}{\sqrt{n_{in}}} $$

That is **Xavier/Glorot** initialization. For ReLU, half the outputs are zeroed,
which halves the variance, so you compensate with a factor of 2:

$$ w \sim \mathcal{N}(0,1)\cdot\sqrt{\frac{2}{n_{in}}} \qquad \textbf{(He initialization)} $$

**The factor of 2 is the entire difference, and it is the difference between a
30-layer ReLU net training and not training.** This is the He et al. 2015 result.

### TODO 2 — the schemes

In [ ]:
def init_weights(fan_in, fan_out, mode, seed=0):
    """-> (fan_in, fan_out) array.

    Use  rng = np.random.RandomState(seed)  and draw ONE randn call so your
    numbers match the reference.

      'zeros'  -> np.zeros
      'small'  -> 0.01 * randn
      'large'  -> 1.0  * randn
      'xavier' -> randn / sqrt(fan_in)
      'he'     -> randn * sqrt(2 / fan_in)
    """
    pass  # TODO

In [ ]:
grade('nb07', 'init_weights', init_weights, verbose=False)

### TODO 3 — measure what happens with depth

In [ ]:
def forward_stats(x, weights, activation):
    """Push x through each weight matrix in turn, applying `activation` after
    each matmul. Return the LIST of activation arrays, one per layer.
    """
    pass  # TODO

In [ ]:
grade('nb07', 'forward_stats', forward_stats, verbose=False)

In [ ]:
iw = use('nb07', 'init_weights', init_weights)
fs = use('nb07', 'forward_stats', forward_stats)

D, L, N = 200, 10, 500
x0 = np.random.RandomState(0).randn(N, D)
relu = lambda t: np.maximum(0, t)

def sweep(activation, modes):
    rows = {}
    for m in modes:
        ws = [iw(D, D, m, seed=i) for i in range(L)]
        rows[m] = [a.std() for a in fs(x0, ws, activation)]
    return rows

tanh_rows = sweep(np.tanh, ['small', 'large', 'xavier'])
relu_rows = sweep(relu,    ['small', 'xavier', 'he'])

print("activation std by layer, tanh:")
for m, v in tanh_rows.items():
    print("  %-7s %s" % (m, " ".join("%.4f" % s for s in v)))
print("\nactivation std by layer, ReLU:")
for m, v in relu_rows.items():
    print("  %-7s %s" % (m, " ".join("%.4f" % s for s in v)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
layers = np.arange(1, L + 1)
for ax, rows, title in [(axes[0], tanh_rows, "tanh"), (axes[1], relu_rows, "ReLU")]:
    plot_series(ax, layers, list(rows.items()),
                "Activation std by depth — %s" % title, "layer",
                "std of activations", direct_labels=False)
    ax.set_yscale('log')
    ax.set_ylim(1e-7, 3)
plt.show()

### Read the two failure modes

- **`small` (0.01·randn)**: the std collapses toward zero, an order of magnitude
  per layer. By layer 6 the activations are numerically zero — and since the
  gradient flowing back is proportional to the activations, **no gradient
  reaches the early layers**. The network is dead.
- **`large` (1.0·randn) with tanh**: std pins near 1.0, which looks fine until
  you realise *every unit is saturated at ±1*. The local gradient
  $1 - \tanh^2$ is then ~0. Also dead, by the opposite route.
- **`xavier` with tanh**: decays gently rather than collapsing. Xavier's
  derivation assumes a linear activation; tanh shrinks its input a little, so
  some decay remains. Good enough in practice, and the standard choice for tanh.
- **`xavier` with ReLU**: decays clearly — this is the halved variance.
- **`he` with ReLU**: holds near constant across all 10 layers. **This is the
  one you want.**

> **The practical rule:** ReLU network → He init (`sqrt(2/fan_in)`). tanh →
> Xavier. In PyTorch this is `nn.init.kaiming_normal_` and
> `nn.init.xavier_normal_`, and the default for `nn.Linear`/`nn.Conv2d` is
> already a Kaiming-uniform variant — so you mostly get it for free, but you
> should know what you are getting.

## 3. Batch normalization

Initialization gives you healthy activations **at step 0**. Then training moves
the weights and the scales drift again. BatchNorm's idea: stop hoping, and
**normalize explicitly at every layer, on every step**.

For each feature $j$, over the minibatch:

$$ \hat{x}_j = \frac{x_j - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}} \qquad y_j = \gamma_j \hat{x}_j + \beta_j $$

The learnable $\gamma, \beta$ matter more than they look. Forcing every layer to
be exactly zero-mean unit-variance would be a real constraint on what the
network can express — so BatchNorm normalizes and then hands the network the
ability to undo it. With $\gamma = \sqrt{\sigma^2+\epsilon}$ and $\beta = \mu$
the layer is the identity. **The network can learn not to normalize.**

### Train and test behave differently — this is the #1 BatchNorm bug

At training time you use the **batch's** statistics. At test time there may be
no batch (a single image), and in any case your prediction must not depend on
which other images happen to be alongside it. So test mode uses a **running
average** of the statistics gathered during training.

Forgetting to switch modes gives you a model that scores well in the training
loop and badly in production. In PyTorch this is `model.train()` /
`model.eval()`, and it is the same bug.

### TODO 4 — `batchnorm_forward`

In [ ]:
def batchnorm_forward(x, gamma, beta, bn_param):
    """x (N, D).  bn_param: 'mode' ('train'|'test'), 'eps', 'momentum',
    'running_mean', 'running_var'.   -> (out, cache)

    train: use the batch mean/var; also update the running estimates with
             running = momentum * running + (1 - momentum) * batch
           and cache whatever backward needs.
    test:  use running_mean / running_var; cache is None.

    Write the updated running stats back into bn_param before returning.
    """
    mode = bn_param['mode']
    eps = bn_param.get('eps', 1e-5)
    momentum = bn_param.get('momentum', 0.9)
    N, D = x.shape
    running_mean = bn_param.get('running_mean', np.zeros(D, dtype=x.dtype))
    running_var = bn_param.get('running_var', np.zeros(D, dtype=x.dtype))

    out, cache = None, None
    # TODO

    bn_param['running_mean'] = running_mean
    bn_param['running_var'] = running_var
    return out, cache

In [ ]:
grade('nb07', 'batchnorm_forward', batchnorm_forward, verbose=False)

In [ ]:
bf = use('nb07', 'batchnorm_forward', batchnorm_forward)

rng = np.random.RandomState(0)
x = rng.randn(200, 4) * 3.0 + 7.0          # badly scaled and off-centre
gamma, beta = rng.randn(4) + 1.0, rng.randn(4)
bp = {'mode': 'train', 'eps': 1e-5, 'momentum': 0.9,
      'running_mean': np.zeros(4), 'running_var': np.ones(4)}
out, cache = bf(x, gamma, beta, bp)

print("input   mean %s\n        std  %s" % (np.round(x.mean(0), 3), np.round(x.std(0), 3)))
print("output  mean %s\n        std  %s" % (np.round(out.mean(0), 3), np.round(out.std(0), 3)))
print("\nbeta    %s   <- output mean should match this" % np.round(beta, 3))
print("gamma   %s   <- output std should match this" % np.round(gamma, 3))

In [ ]:
# The running estimates should converge to the true data statistics.
bp2 = {'mode': 'train', 'eps': 1e-5, 'momentum': 0.9,
       'running_mean': np.zeros(4), 'running_var': np.ones(4)}
r2 = np.random.RandomState(1)
for _ in range(200):
    bf(r2.randn(50, 4) * 3.0 + 7.0, np.ones(4), np.zeros(4), bp2)
print("running_mean %s   (true 7.0)" % np.round(bp2['running_mean'], 3))
print("running_var  %s   (true 9.0)" % np.round(bp2['running_var'], 3))

# and test mode must use them, not the batch
x_test = r2.randn(400, 4) * 3.0 + 7.0
bp2['mode'] = 'test'
out_test, cache_test = bf(x_test, np.ones(4), np.zeros(4), bp2)
print("\ntest-mode output mean %s (near 0, using RUNNING stats)"
      % np.round(out_test.mean(0), 3))
print("test-mode cache is None:", cache_test is None)

### TODO 5 — `batchnorm_backward`

This is the hardest derivation in the course, because $\mu$ and $\sigma^2$ both
depend on **every** $x_i$ — so each input influences every output in the batch.
The branching rule from notebook 04 applies at full strength.

You can grind it out node-by-node, or differentiate on paper and simplify. The
simplified result:

$$ \frac{\partial L}{\partial x} = \frac{1}{N\sigma}\left( N \frac{\partial L}{\partial \hat{x}} - \sum_k \frac{\partial L}{\partial \hat{x}_k} - \hat{x}\sum_k \frac{\partial L}{\partial \hat{x}_k}\hat{x}_k \right) $$

with $\sigma = \sqrt{\text{var} + \epsilon}$ and
$\frac{\partial L}{\partial \hat{x}} = \text{dout} \cdot \gamma$.

The three terms each have a meaning: the direct path, the correction because
you shifted the mean, and the correction because you scaled by the variance.

`dgamma` and `dbeta` are the easy ones — sum over the batch, as always for a
broadcast parameter.

In [ ]:
def batchnorm_backward(dout, cache):
    """-> (dx, dgamma, dbeta)"""
    pass  # TODO

In [ ]:
grade('nb07', 'batchnorm_backward', batchnorm_backward, verbose=False)

In [ ]:
bb = use('nb07', 'batchnorm_backward', batchnorm_backward)

xs = rng.randn(8, 5) * 3.0 + 7.0
gs, bs = rng.randn(5) + 1.0, rng.randn(5)
bps = {'mode': 'train', 'eps': 1e-5, 'momentum': 0.9,
       'running_mean': np.zeros(5), 'running_var': np.ones(5)}
o, c = bf(xs, gs, bs, dict(bps))
d = rng.randn(*o.shape)
dx, dgamma, dbeta = bb(d, c)

print("dx     rel error %.3e" % rel_error(dx, eval_numerical_gradient_array(
    lambda t: bf(t, gs, bs, dict(bps))[0], xs.copy(), d)))
print("dgamma rel error %.3e" % rel_error(dgamma, eval_numerical_gradient_array(
    lambda t: bf(xs, t, bs, dict(bps))[0], gs.copy(), d)))
print("dbeta  rel error %.3e" % rel_error(dbeta, eval_numerical_gradient_array(
    lambda t: bf(xs, gs, t, dict(bps))[0], bs.copy(), d)))

### What BatchNorm actually buys

- **You can use much higher learning rates.** Usually the largest practical win.
- **Initialization stops being critical.** It still helps, but a bad scale gets
  corrected on the first forward pass instead of compounding.
- **It regularizes.** Each example's normalization depends on which other
  examples share its batch, which injects noise — this is why BatchNorm often
  lets you reduce dropout.
- **The bias before it is redundant.** BatchNorm subtracts the mean, which
  removes any constant you just added. `nn.Conv2d(..., bias=False)` before a
  BatchNorm is not an optimization detail, it is the correct thing to write.

And the cost:

- **Batch-size dependence.** With batch size 2 the statistics are garbage. This
  is why LayerNorm (normalize over features, per example) took over in
  Transformers, where sequence batches are small and variable.
- **Train/test skew**, as above.

In [ ]:
# Demonstrate the headline claim: BatchNorm tolerates learning rates that
# destroy a plain network. Deep tanh stack, measure activation health.
def run_depth(mode_init, use_bn, scale, L=10, D=100, N=256, seed=0):
    r = np.random.RandomState(seed)
    h = r.randn(N, D)
    stds = []
    bnp = {'mode': 'train', 'eps': 1e-5, 'momentum': 0.9,
           'running_mean': np.zeros(D), 'running_var': np.ones(D)}
    for i in range(L):
        w = iw(D, D, mode_init, seed=i) * scale
        h = h.dot(w)
        if use_bn:
            h, _ = bf(h, np.ones(D), np.zeros(D), dict(bnp))
        h = np.tanh(h)
        stds.append(h.std())
    return stds

fig, ax = plt.subplots(figsize=(6.4, 3.6))
series = [
    ("no BN, scale 0.5", run_depth('xavier', False, 0.5)),
    ("no BN, scale 3.0", run_depth('xavier', False, 3.0)),
    ("with BN, scale 0.5", run_depth('xavier', True, 0.5)),
    ("with BN, scale 3.0", run_depth('xavier', True, 3.0)),
]
plot_series(ax, np.arange(1, 11), series,
            "BatchNorm makes depth insensitive to weight scale",
            "layer", "activation std", direct_labels=False)
ax.set_yscale('log')
plt.show()

for nm, v in series:
    print("%-20s layer1 %.4f -> layer10 %.4f" % (nm, v[0], v[-1]))

Without BatchNorm the two scales diverge wildly by layer 10. With it, both
settle into the same healthy band regardless of how the weights were
initialized. That insensitivity is the practical reason BatchNorm was such a
big deal.

## 4. Recap

1. Name sigmoid's three problems. Which does ReLU fix?
2. What is the "zig-zag" consequence of non-zero-centred activations?
3. What is a dead ReLU and why can it never recover?
4. Why can you not initialize all weights to zero?
5. Derive $\text{Var}(w) = 1/n_{in}$ from $\text{Var}(y) = n\text{Var}(w)\text{Var}(x)$.
6. Why does He init have a 2 in it and Xavier not?
7. Init `0.01*randn` and init `1.0*randn` both fail with tanh. Describe the two
   *different* failure modes.
8. Why does BatchNorm have learnable $\gamma$ and $\beta$ at all?
9. Exactly what differs between BatchNorm at train and at test time, and what
   is the symptom of getting it wrong?
10. Why is `bias=False` correct on a conv layer immediately before BatchNorm?
11. Why did Transformers move to LayerNorm?

In [ ]:
# show_solution('nb07', 'init_weights')
# show_solution('nb07', 'batchnorm_forward')
# show_solution('nb07', 'batchnorm_backward')